installing the dependencies

In [7]:
!pip install fastapi
!pip install uvicorn
!pip install pickle5
!pip install pydantic
!pip install scikit-learn
!pip install requests
!pip install pypi-json
!pip install pyngrok
!pip install nest-asyncio

  Using cached pickle5-0.0.11.tar.gz (132 kB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pickle5
  Running setup.py clean for pickle5
Failed to build pickle5
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pickle5)


In [8]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import json
import uvicorn
from pyngrok import ngrok
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio

In [9]:
app = FastAPI()

In [10]:
origins = ["*"]
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [11]:
class ModelInput(BaseModel):
  age: int
  sex: int
  bmi: float
  children: int
  smoker: int
  region: int

In [12]:
#loading the saved model
insurance_model = pickle.load(open('insurance_model.sav','rb'))

In [13]:
@app.get("/")
def home():
    return {"message": "Insurance API is running"}

In [14]:
@app.post("/insurance_prediction")
def insurance_pred(input_parameters: ModelInput):
    input_list = [
        input_parameters.age,
        input_parameters.sex,
        input_parameters.bmi,
        input_parameters.children,
        input_parameters.smoker,
        input_parameters.region
    ]

    prediction = insurance_model.predict([input_list])

    return {"predicted_insurance_cost": float(prediction[0])}

In [15]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn

# Fix Colab event loop
nest_asyncio.apply()

# Add ngrok token
token = "your token here"
ngrok.set_auth_token(token)

# Create public URL
ngrok_tunnel = ngrok.connect(8000)
print("Public URL:", ngrok_tunnel.public_url)

# Run FastAPI app
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [9688]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: https://dancing-yapping-crucial.ngrok-free.dev
INFO:     218.208.8.98:0 - "POST /insurance_prediction HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9688]
